# Kmp: once and for all

## LPS as the heart of KMP

LPS = longest **proper** prefix which is alos a suffix
And this lps we need for **every prefix**

A trivial case: lps(0) = 0 => proper prefix

Now for all the rest, say we are at $i$ index and need lps(i), so basically I'm processing this at the level of "first i chars"

**Where to place fingers?**

First look at the substring of first i chars, that's all we will be working with now.

Don't place a finger at the start and then another at the end and try to move them closer -- that's a palindrom.

Instead, one finger at the start, and the other would be at "candidate" suffix starts. Just place it ANYWHERE ( of couse it has to start at 1 index in the suffix due to proper ).
Now move both fingers in the same direction, matching characters, till you can. What you want is to be able to reach to the end with the right finger.

That is the lps.

## string is `ababac`, how do we handle a mismatch?

| Index | Substring so far | LPS |
| ----- | ---------------- | --- |
| 0     | `a`              | 0   |
| 1     | `ab`             | 0   |
| 2     | `aba`            | 1   |
| 3     | `abab`           | 2   |
| 4     | `ababa`          | 3   |

We are at index 5, till index 4, we have "aba" matching with "aba".
Natrually, try to extend first => "abab" with "abac", that's a mismatch. Unfortunate.

**What information do I have still**?

Well, the mismatch is useless to me. _had_ it matches, what I was trying to do is use the BEST candidate I had and try to extend it, that didn't work but what about the second best candidates? the third best?

If I did not have "longest" in lps, then there would be other prefixes.

"aba" vs "aba" at index 4 case ( string being "ababa" ), another possible but not longest PS ( not L ) will be just "a", so naturally, given that we were trying to extend the longest ps and that failed, try to extend the 2nd best ps then third best and so on.

### How to find the 2nd best?

Convince yourself that it is will be of len = lps[length of best lps - 1]

Why?

"aba" matches "aba", so still WANT the prefix of thw whole string at index 5, but we cannot continue that's known by the mismatch. Is there some prefix of this "aba" that I can match still? Note that by the structure of things, I want lps["aba"], that is exacly the prefix as well, so it's just lps[2].

> it's more intuitive to think in terms of prefix spans and indexes, that 2 is indeed len of best lps - 1, but what we want is lps["aba"] as that makes more sense to me

### What if another mismatch?

You keep on falling back, again and again and again.

# KMP is two phase. First is building the LPS

All that we covered above is just the lps.

In [1]:
def build_lps(pattern):
    n = len(pattern)
    lps = [0] * n

    length = 0
    # as lps(0) = 0 so we start at 1
    i = 1 

    while i < n:
        if pattern[i] == pattern[length]:
            # great! looks like we can extend the last match
            length += 1
            lps[i] = length
            i += 1

        # oh, well looks like we have to fallback
        # but can we even fallback?
        elif length > 0:
            # yes we can, this is the new length, use the same i and redo it all
            length = lps[length - 1]
        # no we can't it's all 0 and we move on
        else:
            lps[i] = 0
            i += 1

    return lps


# Test
pattern = "ababac"
print(build_lps(pattern))  # [0, 0, 1, 2, 3, 0]

[0, 0, 1, 2, 3, 0]


## But wait a minute, can this be all?

What if you make string as $pattern|text$

If all you wanted to do fast substring search then by definion of lps, any index i that starts in the text part and has lps(i) == len(pattern) means that the suffix trailing at text(...i) matches pattern, that's my substring.

**In general, don't bother doing a separate kmp at all**
All it costs is just some extra space to store the concatenated string.

# It's still worth it to know the KMP step

and it's quite similar given lps is the bulk of the logic

In [ ]:
def kmp(text, pattern):
    if not pattern:
        return 0

    lps = build_lps(pattern)

    i = 0  # text pointer
    j = 0  # pattern pointer

    while i < len(text):
        # great! similar to lps, if we can just match and extend that's good
        if text[i] == pattern[j]:
            i += 1
            j += 1

            # even better that our match found the whole thing
            if j == len(pattern):
                return i - j  # Found!

        # but if we didn't match
        # similar to lps, is there a fallback? it can only be if there was SOMETHING
        # in the pattern
        elif j > 0:
            # let's try a smaller pattern and see if we match from there
            j = lps[j - 1]

        # well if we are here, j was 0 already so all we can do is increment i
        else:
            i += 1

    return -1


text = "abababac"
pattern = "ababac"

print(kmp(text, pattern))  # 2

2
